# Hopf-GAE: Joint-Bottleneck Normative Graph Autoencoder

## Coupled Edge-Node Reconstruction with Physics-Only Scoring

---

**Changes from v5:**

| Change | Rationale |
|--------|-----------|
| Edge decoders use **z** (trainable) not h (frozen) | v5 edge branch was gradient-isolated from node pathway; coupling forces z to encode connectivity |
| Edge input: **[z_i ∥ z_j]** concatenation | Asymmetric for directed edges (MVAR), unlike \|h_i − h_j\| |
| Anomaly score on **physics-only** (a, ω, χ²) | 7-feature scoring diluted signal (d=3.50 vs 3.86 physics-only); connectivity shapes z during training but does not enter the score |
| Reconstruct all 7 features during training | Expanded targets shape the latent space; physics-only scoring eliminates dilution |
| Default **d_z = 6** | Wider bottleneck needed to jointly encode dynamics + connectivity |
| **Coupling ablation** | Direct comparison: decoupled vs coupled with matched seeds |

**Kernel:** Python 3.10+

---

# S0 — Libraries, Configuration, and Overrides

In [1]:
# =============================================================================
# S0 --- LIBRARIES AND CONFIGURATION
# =============================================================================

import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

import copy
import time
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy import stats as sp_stats
from scipy.spatial.distance import mahalanobis
from sklearn.model_selection import train_test_split
from statsmodels.stats.multitest import multipletests
from torch_geometric.loader import DataLoader as PyGDataLoader
from torch_geometric.nn import global_mean_pool

# Project modules
from config import *
from models import *
from utils import *

# ── v6 overrides ────────────────────────────────────────────────────────
LATENT_DIM = 6  # wider bottleneck for joint node+edge encoding

# Training reconstruction: all 7 features
RECON_FEATURE_WEIGHTS_TRAIN = torch.tensor([2.0, 1.0, 1.0, 0.5, 0.5, 0.5, 0.5])
N_FEATURES_OUT = 7

# Scoring: physics-only (first 3 features)
RECON_FEATURE_WEIGHTS_SCORE = torch.tensor([2.0, 1.0, 1.0])
N_PHYSICS_FEATURES = 3

# Denoising config
NOISE_SIGMA = 0.1
Z_DROPOUT = 0.3
LAMBDA_GRAPH = 0.1

# HC holdout
HC_HOLDOUT_FRAC = 0.15

# Outlier threshold
OUTLIER_N_SD = 5

# Permutations and bootstrap
N_PERMUTATIONS = 10000
N_PERMS = 10000
N_BOOT = 10000

# Seed sweep
N_SEED_RUNS = 10

# Excluded sessions
EXCLUDED_SESSIONS = {("LS_E4209", "rest2")}

seed_everything(SEED)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_V6 = RESULTS_DIR / "v6"
RESULTS_V6.mkdir(parents=True, exist_ok=True)

log.info('v6 configuration loaded. LATENT_DIM=%d, scoring=physics-only(3feat), training=7feat',
         LATENT_DIM)

[11:21:38] v6 configuration loaded. LATENT_DIM=6, scoring=physics-only(3feat), training=7feat


# S1 — Coupled GAE Architecture

The single architectural change: edge decoders operate on **z** (trainable, d_z-dim)
via concatenation `[z_i ∥ z_j]`, instead of on **h** (frozen, 32-dim) via `|h_i − h_j|`.
This forces the edge reconstruction loss to shape z, coupling connectivity and dynamics
in the latent representation.

In [2]:
# =============================================================================
# S1 --- HopfGAE v6: COUPLED EDGE-NODE BOTTLENECK
# =============================================================================


class HopfGAEv6(nn.Module):
    """
    Denoising Graph Autoencoder with COUPLED edge-node bottleneck.

    Key difference from v5 (HopfGAE):
      Edge decoders use z (trainable) not h (frozen).
      Input: [z_i || z_j] (concatenation, 2*latent_dim) for directed asymmetry.
      Gradient path: edge_loss -> edge_decoder -> z -> fc_z (trainable).

    Encoder (frozen): conv1 -> ELU -> conv2 -> ELU -> + residual -> h (32-dim)
    Bottleneck:       h -> Linear -> z (latent_dim, deterministic, trainable)
    Denoising:        x_noisy = x + sigma * N(0,I) during training
    Dropout:          z -> Dropout(p) during training (node decoder only)
    Node decoder:     Dropout(z) -> Linear -> 7 features
    Edge decoders:    MLP([z_i || z_j]) -> P(edge_ij) per relation (NO dropout on z)
    """

    def __init__(self, encoder_model, latent_dim=6, n_features_out=7,
                 noise_sigma=0.1, z_dropout=0.3):
        super().__init__()

        # Freeze the encoder
        self.encoder = encoder_model
        for param in self.encoder.parameters():
            param.requires_grad = False

        hidden_dim = encoder_model.hidden_dim

        # Deterministic bottleneck: h -> z
        self.fc_z = nn.Linear(hidden_dim, latent_dim)

        # Dropout on z before node decoder only
        self.z_dropout = nn.Dropout(p=z_dropout)

        # Linear node decoder: z -> 7 reconstructed features
        self.node_decoder = nn.Linear(latent_dim, n_features_out)

        # --- v6 change: edge decoders use z, not h ---
        # Input: [z_i || z_j] = 2 * latent_dim (concatenation for directed asymmetry)
        edge_input_dim = 2 * latent_dim
        edge_hidden = max(latent_dim, 8)
        self.edge_decoders = nn.ModuleDict({
            rel: nn.Sequential(
                nn.Linear(edge_input_dim, edge_hidden), nn.ELU(),
                nn.Linear(edge_hidden, 1),
            )
            for rel in ["plv", "sc", "mvar"]
        })

        self.latent_dim = latent_dim
        self.hidden_dim = hidden_dim
        self.noise_sigma = noise_sigma
        self.n_features_out = n_features_out

    def encode(self, data):
        """Run frozen encoder, return h and deterministic z."""
        if not hasattr(data, "batch") or data.batch is None:
            data.batch = torch.zeros(data.x.size(0), dtype=torch.long, device=data.x.device)

        with torch.no_grad():
            enc_out = self.encoder(data)
        h = enc_out["node_embeddings"]

        z = self.fc_z(h)
        return h, z, data.batch

    def decode_nodes(self, z):
        """Decode from z with dropout."""
        z_d = self.z_dropout(z)
        return self.node_decoder(z_d)

    def decode_edges(self, z, edge_index, relation):
        """v6: MLP edge decoder using [z_i || z_j] (trainable z, not frozen h)."""
        if relation not in self.edge_decoders:
            return None
        src, dst = edge_index
        # Concatenation preserves direction: [z_src || z_dst] != [z_dst || z_src]
        edge_feat = torch.cat([z[src], z[dst]], dim=-1)
        return torch.sigmoid(self.edge_decoders[relation](edge_feat).squeeze(-1))

    def forward(self, data):
        # Denoising: inject noise on encoder input during training
        if self.training and self.noise_sigma > 0:
            data = data.clone()
            noise = self.noise_sigma * torch.randn_like(data.x)
            data.x = data.x + noise

        h, z, batch = self.encode(data)
        node_recon = self.decode_nodes(z)

        return {
            "node_recon": node_recon,
            "h": h,
            "z": z,
            "batch": batch,
        }


class GAELossV6(nn.Module):
    """
    v6 loss: identical to v5 GAELoss except edge decoders receive z, not h.

    L = L_node_weighted + lambda_graph * L_graph + lambda_edge * L_edge
    """

    def __init__(self, lambda_edge=0.5, lambda_graph=0.1,
                 feature_weights=None):
        super().__init__()
        self.lambda_edge = lambda_edge
        self.lambda_graph = lambda_graph

        if feature_weights is not None:
            self.register_buffer("feature_weights", feature_weights)
        else:
            self.register_buffer("feature_weights", RECON_FEATURE_WEIGHTS)

    def forward(self, result, data, gae_model=None):
        loss_dict = {}

        # ── Node reconstruction (7 features, weighted) ───────────────────
        target_nodes = data.recon_target
        recon = result["node_recon"]
        diff_sq = (recon - target_nodes) ** 2
        w = self.feature_weights.to(diff_sq.device)
        if w.shape[0] < diff_sq.shape[1]:
            w = torch.cat([w, torch.ones(diff_sq.shape[1] - w.shape[0], device=w.device)])
        l_node = (diff_sq * w[:diff_sq.shape[1]]).mean()
        loss_dict["node_recon"] = l_node.item()

        # ── Graph-level reconstruction (mean + std of a) ─────────────────
        batch = result["batch"]
        a_recon = recon[:, 0]
        a_target = target_nodes[:, 0]

        mean_recon = global_mean_pool(a_recon.unsqueeze(1), batch).squeeze(1)
        mean_target = global_mean_pool(a_target.unsqueeze(1), batch).squeeze(1)
        l_graph_mean = F.mse_loss(mean_recon, mean_target)

        a_recon_sq = global_mean_pool((a_recon ** 2).unsqueeze(1), batch).squeeze(1)
        a_target_sq = global_mean_pool((a_target ** 2).unsqueeze(1), batch).squeeze(1)
        var_recon = torch.clamp(a_recon_sq - mean_recon ** 2, min=1e-8)
        var_target = torch.clamp(a_target_sq - mean_target ** 2, min=1e-8)
        l_graph_std = F.mse_loss(var_recon.sqrt(), var_target.sqrt())

        l_graph = l_graph_mean + l_graph_std
        loss_dict["graph_level"] = l_graph.item()

        # ── Per-relation edge reconstruction (v6: uses z, not h) ─────────
        z = result["z"]   # <-- v6 change: z instead of h
        l_edge = torch.tensor(0.0, device=z.device)
        n_edge_terms = 0

        if gae_model is not None:
            for rel in ["plv", "sc", "mvar"]:
                ei_key = f"edge_index_{rel}"
                if hasattr(data, ei_key) and getattr(data, ei_key).numel() > 0:
                    ei = getattr(data, ei_key)
                    edge_pred = gae_model.decode_edges(z, ei, rel)  # <-- z not h
                    if edge_pred is not None:
                        edge_true = torch.ones_like(edge_pred)
                        l_edge = l_edge + F.binary_cross_entropy(edge_pred, edge_true)
                        n_edge_terms += 1

                        # Negative edges
                        n_nodes = z.size(0)
                        n_neg = min(ei.size(1), n_nodes * 2)
                        neg_src = torch.randint(0, n_nodes, (n_neg,), device=z.device)
                        neg_dst = torch.randint(0, n_nodes, (n_neg,), device=z.device)
                        neg_ei = torch.stack([neg_src, neg_dst])
                        neg_pred = gae_model.decode_edges(z, neg_ei, rel)
                        if neg_pred is not None:
                            neg_true = torch.zeros_like(neg_pred)
                            l_edge = l_edge + F.binary_cross_entropy(neg_pred, neg_true)
                            n_edge_terms += 1

        if n_edge_terms > 0:
            l_edge = l_edge / n_edge_terms
        loss_dict["edge_recon"] = l_edge.item()

        total = l_node + self.lambda_graph * l_graph + self.lambda_edge * l_edge
        loss_dict["total"] = total.item()

        return total, loss_dict


# Verify architecture
log.info('HopfGAEv6 and GAELossV6 defined.')
log.info('  Edge decoder input: [z_i || z_j] = 2 * %d = %d dims (trainable)',
         LATENT_DIM, 2 * LATENT_DIM)
log.info('  Training targets: 7 features, Scoring targets: 3 physics features')

[11:21:38] HopfGAEv6 and GAELossV6 defined.
[11:21:38]   Edge decoder input: [z_i || z_j] = 2 * 6 = 12 dims (trainable)
[11:21:38]   Training targets: 7 features, Scoring targets: 3 physics features


# S2 — Data Loading

In [3]:
# =============================================================================
# S2 --- DATA LOADING
# =============================================================================

all_data = load_all_data()
ukf_df = all_data["ukf_df"]
plv_all = all_data["plv_all"]
mvar_all = all_data["mvar_all"]
group_df = all_data["group_df"]

[11:21:39] UKF: 8205 rows, 19 subjects
[11:21:39] PLV: 38 matrices
[11:21:39] MVAR: 38 matrices


# S3 — ROI Metadata and Masks

In [4]:
# =============================================================================
# S3 --- ROI METADATA + MASKS
# =============================================================================

roi_meta, network_assignment, N_NETWORKS = build_roi_meta_and_assignment(ukf_df)

circuit_mask = roi_meta['is_depression_circuit'].values
limbic_mask = (roi_meta['network'] == 'Limbic').values
subcort_mask = (roi_meta['network'] == 'Subcortical').values
lh_mask = roi_meta['roi_name'].str.contains('_LH_|lh|-lh', case=False, regex=True).values
rh_mask = roi_meta['roi_name'].str.contains('_RH_|rh|-rh', case=False, regex=True).values
amyg_mask = roi_meta['roi_name'].str.contains('Amyg', case=False).values

network_masks = {}
for net_name in roi_meta['network'].unique():
    network_masks[net_name] = (roi_meta['network'] == net_name).values

[11:21:39] ROI metadata: 216 ROIs, 8 networks, 69 circuit ROIs


# S4 — Structural Connectivity

In [5]:
# =============================================================================
# S4 --- STRUCTURAL CONNECTIVITY
# =============================================================================

sc_matrix, centroids = load_or_build_sc(roi_meta)
log.info('SC matrix: shape %s, density %.2f%%',
         sc_matrix.shape, 100 * (sc_matrix > 0).sum() / sc_matrix.size)

[11:21:39] SC from atlas centroids: (216, 216)
[11:21:39] SC matrix: shape (216, 216), density 99.54%


# S5 — Simulator and QC

In [6]:
# =============================================================================
# S5 --- SIMULATOR + QC
# =============================================================================

simulator = StuartLandauSimulator(sc_matrix, n_rois=N_ROIS_216, TR=TR, n_TRs=N_VOLS)

log.info('=' * 70)
log.info('DATA QUALITY CONTROL')
log.info('=' * 70)
for sess in sorted(ukf_df['session'].unique()):
    a_vals = ukf_df.loc[ukf_df['session'] == sess, 'a']
    log.info('  %s: n=%d, mean_a=%.4f +/- %.4f, subcritical=%.1f%%',
             sess, len(a_vals), a_vals.mean(), a_vals.std(), (a_vals < 0).mean() * 100)

test_subj = ukf_df['subject'].unique()[0]
test_sess = ukf_df['session'].unique()[0]
test_graph = build_subject_graph(
    test_subj, test_sess, ukf_df, roi_meta,
    plv_mat=np.array(plv_all[f"{test_subj}|{test_sess}"]) if plv_all else None,
    mvar_mat=np.array(mvar_all[f"{test_subj}|{test_sess}"]) if mvar_all else None,
    sc_mat=sc_matrix,
    group=ukf_df[ukf_df['subject'] == test_subj]['group'].iloc[0],
)
log.info('Test graph: %s, x=%s', test_subj, test_graph.x.shape)

[11:21:39] ======================================================================
[11:21:39] DATA QUALITY CONTROL
[11:21:39] ======================================================================
[11:21:39]   rest1: n=4101, mean_a=-0.2854 +/- 0.1573, subcritical=100.0%
[11:21:39]   rest2: n=4104, mean_a=-0.2906 +/- 0.1620, subcritical=100.0%
[11:21:39] Test graph: AC_E5694, x=torch.Size([216, 11])


# S6 — Encoder + Pre-Training

In [7]:
# =============================================================================
# S6 --- ENCODER + SYNTHETIC PRE-TRAINING
# =============================================================================

encoder = HopfEncoder(
    n_node_features=3 + N_NETWORKS, hidden_dim=HIDDEN_DIM, dropout=DROPOUT,
)
log.info('HopfEncoder: %d params', sum(p.numel() for p in encoder.parameters()))

# ── Synthetic data ──
log.info('=' * 70)
log.info('STAGE 1: SYNTHETIC DATA + PRE-TRAINING')
log.info('=' * 70)

seed_everything(SEED)


def generate_synthetic_graphs(sim, n_train, n_val, n_nets, net_assign, sc_mat, seed=SEED):
    syn_train, syn_val = [], []
    for i in range(n_train + n_val):
        result = sim.generate_graph(
            a_mean=np.random.uniform(-0.40, -0.05),
            a_std=np.random.uniform(0.05, 0.20),
            G=np.random.uniform(0.2, 1.0), seed=seed + i,
        )
        nr = sim.n_rois
        x = torch.zeros(nr, 3 + n_nets, dtype=torch.float32)
        x[:, 0] = torch.tensor(result['a_true'], dtype=torch.float32)
        x[:, 1] = torch.tensor(result['omega'] / (2 * np.pi * TR), dtype=torch.float32)
        x[:, 2] = 0.5
        if net_assign is not None:
            for j in range(min(nr, len(net_assign))):
                x[j, 3 + net_assign[j].item()] = 1.0

        ei_plv, ea_plv = matrix_to_edge_index(result['plv'][:nr, :nr], directed=False, top_k_pct=PLV_TOP_K)
        ei_sc, ea_sc = matrix_to_edge_index(sc_mat[:nr, :nr], directed=False, top_k_pct=SC_TOP_K)
        ei_mvar, ea_mvar = matrix_to_edge_index(result['mvar'][:nr, :nr], directed=True, threshold=1e-10)

        data = Data(
            x=x, num_nodes=nr,
            edge_index_plv=ei_plv, edge_attr_plv=ea_plv,
            edge_index_mvar=ei_mvar, edge_attr_mvar=ea_mvar,
            edge_index_sc=ei_sc, edge_attr_sc=ea_sc,
            a_true=torch.tensor(result['a_true'][:nr], dtype=torch.float32),
        )
        (syn_train if i < n_train else syn_val).append(data)
        if (i + 1) % 50 == 0:
            log.info('  Generated %d / %d', i + 1, n_train + n_val)

    log.info('Synthetic dataset: %d train + %d val', len(syn_train), len(syn_val))
    return syn_train, syn_val


syn_train, syn_val = generate_synthetic_graphs(
    simulator, N_SYN, N_VAL_SYN, N_NETWORKS, network_assignment, sc_matrix)

# ── Pre-training loop ──
encoder.train()
criterion_pt = HopfPhysicsLoss(lambda_physics=LAMBDA_PHYSICS, lambda_subcrit=LAMBDA_SUBCRIT)
optimizer_pt = torch.optim.AdamW(encoder.parameters(), lr=PRETRAIN_LR, weight_decay=PRETRAIN_WD)
scheduler_pt = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_pt, T_max=PRETRAIN_EPOCHS)

pt_train_loader = PyGDataLoader(syn_train, batch_size=PRETRAIN_BS, shuffle=True)
pt_val_loader = PyGDataLoader(syn_val, batch_size=max(1, len(syn_val)))
pretrain_history = {'epoch': [], 'train_loss': [], 'val_loss': [], 'r2': []}

for epoch in range(1, PRETRAIN_EPOCHS + 1):
    encoder.train()
    ep_loss, nb_pt = 0.0, 0
    for batch in pt_train_loader:
        optimizer_pt.zero_grad()
        out = encoder(batch)
        loss, _ = criterion_pt(out['a_pred'], batch.a_true)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(encoder.parameters(), max_norm=1.0)
        optimizer_pt.step()
        ep_loss += loss.item(); nb_pt += 1
    scheduler_pt.step()

    encoder.eval()
    with torch.no_grad():
        all_preds, all_trues = [], []
        val_loss_pt = 0.0
        for batch in pt_val_loader:
            out = encoder(batch)
            vl, _ = criterion_pt(out['a_pred'], batch.a_true)
            val_loss_pt += vl.item()
            all_preds.append(out['a_pred'].squeeze().cpu().numpy())
            all_trues.append(batch.a_true.cpu().numpy())

    vp = np.concatenate(all_preds)
    vt = np.concatenate(all_trues)
    ss_res = np.sum((vp - vt) ** 2)
    ss_tot = np.sum((vt - vt.mean()) ** 2)
    r2 = 1 - ss_res / max(ss_tot, 1e-12)

    # Graph-mean R² for v4 comparability
    batch_ids = np.zeros(len(vt), dtype=int)
    if hasattr(batch, 'batch') and batch.batch is not None:
        batch_ids = batch.batch.cpu().numpy()
    ugs = np.unique(batch_ids)
    gm_p = np.array([vp[batch_ids == g].mean() for g in ugs])
    gm_t = np.array([vt[batch_ids == g].mean() for g in ugs])
    ss_r_gm = np.sum((gm_p - gm_t) ** 2)
    ss_t_gm = np.sum((gm_t - gm_t.mean()) ** 2)
    r2_graph = 1 - ss_r_gm / max(ss_t_gm, 1e-12)

    pretrain_history['epoch'].append(epoch)
    pretrain_history['train_loss'].append(ep_loss / max(nb_pt, 1))
    pretrain_history['val_loss'].append(val_loss_pt)
    pretrain_history['r2'].append(r2)

    if epoch % 20 == 0 or epoch == 1:
        log.info('  Epoch %3d | Train %.4f | Val %.4f | R2_roi %.3f | R2_graph %.3f | LR %.1e',
                 epoch, ep_loss / max(nb_pt, 1), val_loss_pt, r2, r2_graph,
                 scheduler_pt.get_last_lr()[0])

log.info('Pre-training complete. R2_roi = %.3f, R2_graph = %.3f', r2, r2_graph)
for lname in ['conv1', 'conv2']:
    conv = getattr(encoder, lname)
    wts = {k: f'{w:.3f}' for k, w in zip(RELATION_KEYS, conv.relation_weights.tolist())}
    log.info('Learned relation weights (%s): %s', lname, wts)

[11:21:39] HopfEncoder: 5485 params
[11:21:39] ======================================================================
[11:21:39] STAGE 1: SYNTHETIC DATA + PRE-TRAINING
[11:21:39] ======================================================================
[11:22:14]   Generated 50 / 220
[11:22:53]   Generated 100 / 220
[11:23:27]   Generated 150 / 220
[11:24:01]   Generated 200 / 220
[11:24:14] Synthetic dataset: 200 train + 20 val
[11:24:15]   Epoch   1 | Train 0.0280 | Val 0.0275 | R2_roi -0.087 | R2_graph -0.257 | LR 3.0e-03
[11:24:30]   Epoch  20 | Train 0.0117 | Val 0.0138 | R2_roi 0.457 | R2_graph 0.974 | LR 2.7e-03
[11:24:46]   Epoch  40 | Train 0.0096 | Val 0.0109 | R2_roi 0.569 | R2_graph 0.971 | LR 2.0e-03
[11:25:03]   Epoch  60 | Train 0.0091 | Val 0.0101 | R2_roi 0.603 | R2_graph 0.957 | LR 1.0e-03
[11:25:19]   Epoch  80 | Train 0.0088 | Val 0.0098 | R2_roi 0.613 | R2_graph 0.962 | LR 2.9e-04
[11:25:35]   Epoch 100 | Train 0.0087 | Val 0.0097 | R2_roi 0.616 | R2_graph 0.964 | LR 

# S7 — HC Data + MDD Graphs + Augmentation

In [8]:
# =============================================================================
# S7 --- HC + MDD DATA + AUGMENTATION
# =============================================================================

log.info('=' * 70)
log.info('STAGE 2: DATA LOADING + AUGMENTATION')
log.info('=' * 70)

hc_graphs, hc_file_info = load_hc_graphs(
    roi_meta, network_assignment, N_NETWORKS, sc_matrix, include_mvar=True)
hc_train_graphs, hc_test_graphs = split_hc_by_subject(hc_graphs, hc_file_info)
hc_train_graphs, hc_holdout_graphs = train_test_split(
    hc_train_graphs, test_size=HC_HOLDOUT_FRAC, random_state=SEED)
log.info('HC split: %d train, %d test, %d holdout',
         len(hc_train_graphs), len(hc_test_graphs), len(hc_holdout_graphs))

empirical_graphs, subjects_list, groups_map = build_empirical_graphs(
    ukf_df, roi_meta, plv_all, mvar_all, sc_matrix)
for key in list(EXCLUDED_SESSIONS):
    if key in empirical_graphs:
        del empirical_graphs[key]
        log.info('EXCLUDED: %s %s', key[0], key[1])
log.info('MDD graphs after exclusion: %d', len(empirical_graphs))


# ── Augmentation (add derived features as recon targets) ──
def augment_graph(graph, net_assign, n_rois=216):
    physics = graph.x[:, :3]
    n_nodes = graph.x.size(0)

    def _ea(g, rel, ne):
        ea = getattr(g, f'edge_attr_{rel}', None)
        if ea is None: return torch.ones(ne)
        ea = ea.float().view(-1)
        return ea if ea.shape[0] == ne else torch.ones(ne)

    s_plv = torch.zeros(n_nodes)
    if hasattr(graph, 'edge_index_plv') and graph.edge_index_plv.numel() > 0:
        ei = graph.edge_index_plv
        s_plv.scatter_add_(0, ei[0], _ea(graph, 'plv', ei.size(1)))
        s_plv = s_plv / max(n_nodes, 1)

    s_mvar_in, s_mvar_out = torch.zeros(n_nodes), torch.zeros(n_nodes)
    if hasattr(graph, 'edge_index_mvar') and graph.edge_index_mvar.numel() > 0:
        ei = graph.edge_index_mvar
        ea = _ea(graph, 'mvar', ei.size(1)).abs()
        s_mvar_in.scatter_add_(0, ei[1], ea)
        s_mvar_out.scatter_add_(0, ei[0], ea)
        s_mvar_in /= max(n_nodes, 1); s_mvar_out /= max(n_nodes, 1)

    plv_within = torch.zeros(n_nodes)
    if hasattr(graph, 'edge_index_plv') and graph.edge_index_plv.numel() > 0:
        ei = graph.edge_index_plv
        ea = _ea(graph, 'plv', ei.size(1))
        na = net_assign[:n_nodes] if net_assign is not None else None
        if na is not None:
            same = (na[ei[0]] == na[ei[1]])
            ws = torch.zeros(n_nodes); wc = torch.zeros(n_nodes)
            ws.scatter_add_(0, ei[0][same], ea[same])
            wc.scatter_add_(0, ei[0][same], torch.ones(same.sum()))
            plv_within = ws / wc.clamp(min=1)

    graph.recon_target = torch.cat([
        physics, s_plv.unsqueeze(1), s_mvar_in.unsqueeze(1),
        s_mvar_out.unsqueeze(1), plv_within.unsqueeze(1),
    ], dim=1)
    return graph


for g in hc_train_graphs + hc_test_graphs + hc_holdout_graphs:
    augment_graph(g, network_assignment)
for g in empirical_graphs.values():
    augment_graph(g, network_assignment)
log.info('All graphs augmented with 7-feature reconstruction targets')

[11:25:35] ======================================================================
[11:25:35] STAGE 2: DATA LOADING + AUGMENTATION
[11:25:35] ======================================================================
[11:25:36] HC MVAR loaded: 295 matrices
[11:25:38] HC UKF from ch5 key: hc_all
[11:25:38] HC CSVs: 295
[11:25:38] HC files parsed: 295 (subjects: 30)
[11:25:38] HC ROI columns matched: 216 / 216
[11:25:50]   Built 50 / 295 HC graphs
[11:26:01]   Built 100 / 295 HC graphs
[11:26:13]   Built 150 / 295 HC graphs
[11:26:24]   Built 200 / 295 HC graphs
[11:26:36]   Built 250 / 295 HC graphs
[11:26:46] HC graphs: 295 / 295
[11:26:46] HC split: train=235 (24 subj), test=60 (6 subj)
[11:26:46] HC split: 199 train, 60 test, 36 holdout
[11:26:47] MDD graphs: 38 (subjects: 19, groups: {'sham': 8, 'active': 11})
[11:26:47] EXCLUDED: LS_E4209 rest2
[11:26:47] MDD graphs after exclusion: 37
[11:26:47] All graphs augmented with 7-feature reconstruction targets


# S8 — Reusable Pipeline Infrastructure

In [9]:
# =============================================================================
# S8 --- REUSABLE PIPELINE FUNCTIONS
# =============================================================================


# ─── Statistical Helpers ─────────────────────────────────────────────────

def cohens_d(a, b):
    pooled = np.sqrt((np.var(a) + np.var(b)) / 2)
    return (np.mean(a) - np.mean(b)) / max(pooled, 1e-12)


def permutation_test_d(a, b, n_perms=N_PERMS, seed=SEED):
    obs_d = cohens_d(a, b)
    combined = np.concatenate([a, b])
    na = len(a)
    rng = np.random.RandomState(seed)
    perm_ds = np.zeros(n_perms)
    for i in range(n_perms):
        idx = rng.permutation(len(combined))
        perm_ds[i] = cohens_d(combined[idx[:na]], combined[idx[na:]])
    return obs_d, (np.abs(perm_ds) >= np.abs(obs_d)).mean(), perm_ds


def bootstrap_ci_d(a, b, n_boot=N_BOOT, seed=SEED):
    rng = np.random.RandomState(seed)
    boot_d = np.zeros(n_boot)
    for i in range(n_boot):
        boot_d[i] = cohens_d(a[rng.randint(0, len(a), len(a))],
                             b[rng.randint(0, len(b), len(b))])
    return np.percentile(boot_d, [2.5, 97.5])


def hypergeom_enrichment(top_k_rois, circ_mask, n_total):
    from scipy.stats import hypergeom
    K, n = circ_mask.sum(), len(top_k_rois)
    k = circ_mask[top_k_rois].sum()
    p = hypergeom.sf(k - 1, n_total, K, n)
    return k, n, (k / n) / (K / n_total) if K > 0 and n > 0 else 0, p


# ─── Scoring Functions (physics-only by default) ────────────────────────

def score_node_physics(gae_model, graph, fw_score=RECON_FEATURE_WEIGHTS_SCORE):
    """Per-ROI reconstruction error on PHYSICS features only (a, ω, χ²)."""
    g_c = prepare_graph_for_batching(graph) if hasattr(graph, 'subject') else graph
    gae_model.eval()
    with torch.no_grad():
        result = gae_model(g_c)
    n_feat = len(fw_score)
    target = g_c.recon_target[:, :n_feat].numpy()
    recon = result['node_recon'][:, :n_feat].numpy()
    diff = np.clip(target - recon, -1e3, 1e3)
    node_err = np.average(diff ** 2, axis=1, weights=fw_score.numpy())
    return node_err, result


def score_node_full(gae_model, graph, fw=RECON_FEATURE_WEIGHTS_TRAIN):
    """Per-ROI reconstruction error on ALL 7 features."""
    g_c = prepare_graph_for_batching(graph) if hasattr(graph, 'subject') else graph
    gae_model.eval()
    with torch.no_grad():
        result = gae_model(g_c)
    target = g_c.recon_target.numpy()
    recon = result['node_recon'].numpy()
    diff = np.clip(target - recon, -1e3, 1e3)
    w = fw.numpy()
    if w.shape[0] < diff.shape[1]:
        w = np.concatenate([w, np.ones(diff.shape[1] - w.shape[0])])
    return np.average(diff ** 2, axis=1, weights=w[:diff.shape[1]]), result


def score_edge_recon(gae_model, graph, result_dict):
    """Per-ROI edge reconstruction error using z (v6) or h (v5)."""
    z = result_dict['z']
    n_nodes = z.size(0)
    g_c = prepare_graph_for_batching(graph) if hasattr(graph, 'subject') else graph
    total_err, total_count = np.zeros(n_nodes), np.zeros(n_nodes)
    for rel in ['plv', 'sc', 'mvar']:
        ei_key = f'edge_index_{rel}'
        if hasattr(g_c, ei_key) and getattr(g_c, ei_key).numel() > 0:
            ei = getattr(g_c, ei_key)
            pred = gae_model.decode_edges(z, ei, rel)
            if pred is not None:
                bce = -torch.log(pred.clamp(min=1e-7)).cpu().detach().numpy()
                src, dst = ei[0].cpu().numpy(), ei[1].cpu().numpy()
                np.add.at(total_err, src, bce); np.add.at(total_err, dst, bce)
                np.add.at(total_count, src, 1); np.add.at(total_count, dst, 1)
    return total_err / np.maximum(total_count, 1)


def score_graphs(gae_model, graphs, scoring_fn=score_node_physics):
    """Score a list of graphs. Returns (per_graph_scores, per_graph_roi_errors)."""
    scores, roi_errors = [], []
    for g in graphs:
        err, _ = scoring_fn(gae_model, g)
        scores.append(err.mean())
        roi_errors.append(err)
    return np.array(scores), roi_errors


# ─── Training Function ──────────────────────────────────────────────────

def train_gae_v6(encoder_model, hc_train, hc_test,
                 latent_dim=LATENT_DIM, n_features_out=N_FEATURES_OUT,
                 noise_sigma=NOISE_SIGMA, z_dropout=Z_DROPOUT,
                 lambda_edge=LAMBDA_EDGE, lambda_graph=LAMBDA_GRAPH,
                 feature_weights=RECON_FEATURE_WEIGHTS_TRAIN,
                 epochs=GAE_EPOCHS, lr=GAE_LR, wd=GAE_WD, bs=GAE_BS,
                 use_coupled=True, verbose=True):
    """Train a GAE model. use_coupled=True → v6, False → v5 (decoupled)."""
    if use_coupled:
        gae = HopfGAEv6(encoder_model, latent_dim=latent_dim,
                         n_features_out=n_features_out,
                         noise_sigma=noise_sigma, z_dropout=z_dropout)
    else:
        gae = HopfGAE(encoder_model, latent_dim=latent_dim,
                       n_features_out=n_features_out,
                       noise_sigma=noise_sigma, z_dropout=z_dropout)

    crit = GAELossV6(lambda_edge=lambda_edge, lambda_graph=lambda_graph,
                     feature_weights=feature_weights) if use_coupled else \
           GAELoss(lambda_edge=lambda_edge, lambda_graph=lambda_graph,
                   feature_weights=feature_weights)

    trainable = [p for p in gae.parameters() if p.requires_grad]
    n_train_p = sum(p.numel() for p in trainable)
    if verbose:
        log.info('GAE (%s): trainable=%d, d_z=%d, σ=%.2f, drop=%.1f, λ_edge=%.2f',
                 'coupled' if use_coupled else 'decoupled', n_train_p,
                 latent_dim, noise_sigma, z_dropout, lambda_edge)

    opt = torch.optim.AdamW(trainable, lr=lr, weight_decay=wd)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)

    train_loader = PyGDataLoader(hc_train, batch_size=bs, shuffle=True)
    test_loader = PyGDataLoader(hc_test, batch_size=max(1, len(hc_test)))

    history = {'epoch': [], 'train_loss': [], 'test_loss': [],
               'node_recon': [], 'edge_recon': []}

    for epoch in range(1, epochs + 1):
        gae.train()
        el, en, ee, nb = 0., 0., 0., 0
        for bd in train_loader:
            opt.zero_grad()
            res = gae(bd)
            loss, ld = crit(res, bd, gae_model=gae)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(trainable, max_norm=1.0)
            opt.step()
            el += ld['total']; en += ld['node_recon']; ee += ld['edge_recon']; nb += 1
        sched.step()

        gae.eval()
        with torch.no_grad():
            tl = sum(crit(gae(tb), tb, gae_model=gae)[0].item() for tb in test_loader)

        history['epoch'].append(epoch)
        history['train_loss'].append(el / max(nb, 1))
        history['test_loss'].append(tl)
        history['node_recon'].append(en / max(nb, 1))
        history['edge_recon'].append(ee / max(nb, 1))

        if verbose and (epoch % 50 == 0 or epoch == 1):
            log.info('  Epoch %3d | Train %.5f | Test %.5f | Node %.5f | Edge %.5f',
                     epoch, el / max(nb, 1), tl, en / max(nb, 1), ee / max(nb, 1))

    if verbose:
        log.info('GAE training complete.')
    return gae, history


# ─── Full Experiment Runner ──────────────────────────────────────────────

def run_experiment(encoder_model, hc_train, hc_test, hc_holdout,
                   emp_graphs, subj_list, grp_map,
                   latent_dim=LATENT_DIM, noise_sigma=NOISE_SIGMA,
                   z_dropout=Z_DROPOUT, lambda_edge=LAMBDA_EDGE,
                   lambda_graph=LAMBDA_GRAPH, epochs=GAE_EPOCHS,
                   outlier_n_sd=OUTLIER_N_SD,
                   scoring_fn=score_node_physics,
                   use_coupled=True, verbose=True, seed=SEED):
    """Run full pipeline. Returns metrics dict."""
    seed_everything(seed)
    enc = copy.deepcopy(encoder_model)
    t0 = time.time()

    gae, history = train_gae_v6(
        enc, hc_train, hc_test, latent_dim=latent_dim,
        noise_sigma=noise_sigma, z_dropout=z_dropout,
        lambda_edge=lambda_edge, lambda_graph=lambda_graph,
        epochs=epochs, use_coupled=use_coupled, verbose=verbose)

    train_time = time.time() - t0

    # Score
    hc_tr_s, hc_tr_r = score_graphs(gae, hc_train, scoring_fn)
    hc_te_s, hc_te_r = score_graphs(gae, hc_test, scoring_fn)
    hc_ho_s, _ = score_graphs(gae, hc_holdout, scoring_fn)

    hc_all = np.concatenate([hc_tr_s, hc_te_s])
    mu, sd = hc_all.mean(), max(hc_all.std(), 1e-12)

    hc_te_z = (hc_te_s - mu) / sd
    hc_ho_z = (hc_ho_s - mu) / sd

    mdd_r1 = [emp_graphs[(s, 'rest1')] for s in subj_list if (s, 'rest1') in emp_graphs]
    mdd_r1_s, mdd_r1_r = score_graphs(gae, mdd_r1, scoring_fn)
    mdd_r1_z = (mdd_r1_s - mu) / sd

    # Outlier
    thresh = mu + outlier_n_sd * sd
    outlier_mask = mdd_r1_s > thresh

    # HC–MDD
    d_sep = cohens_d(mdd_r1_z, hc_te_z)
    _, p_welch = sp_stats.ttest_ind(mdd_r1_z, hc_te_z, equal_var=False)
    hc_tr_z = (hc_tr_s - mu) / sd
    _, p_overfit = sp_stats.ttest_ind(hc_tr_z, hc_te_z, equal_var=False)
    fp_rate = (hc_ho_s > thresh).sum() / max(len(hc_ho_s), 1)

    # Intervention
    analysis_subjs = [s for i, s in enumerate(subj_list) if not outlier_mask[i]]
    intervention = {}
    scales = {'whole_brain': np.ones(len(roi_meta), dtype=bool),
              'circuit': circuit_mask, 'limbic': limbic_mask, 'subcortical': subcort_mask}

    for sn, mask in scales.items():
        act_d, sha_d = [], []
        for subj in analysis_subjs:
            r1k, r2k = (subj, 'rest1'), (subj, 'rest2')
            if r1k not in emp_graphs or r2k not in emp_graphs: continue
            e1, _ = scoring_fn(gae, emp_graphs[r1k])
            e2, _ = scoring_fn(gae, emp_graphs[r2k])
            delta = e2[mask].mean() - e1[mask].mean()
            grp = grp_map.get(subj, '').lower()
            (act_d if grp in ('active', 'experimental') else sha_d).append(delta)

        if len(act_d) >= 2 and len(sha_d) >= 2:
            act_d, sha_d = np.array(act_d), np.array(sha_d)
            d_int = cohens_d(act_d, sha_d)
            _, p_int = sp_stats.ttest_ind(act_d, sha_d, equal_var=False)
            comb = np.concatenate([act_d, sha_d]); na = len(act_d)
            rng = np.random.RandomState(seed)
            perm = np.array([cohens_d(s[:na], s[na:]) for s in
                             [rng.permutation(comb) for _ in range(N_PERMS)]])
            p_perm = (np.abs(perm) >= np.abs(d_int)).mean()
            intervention[sn] = {
                'd': d_int, 'p': p_int, 'perm_p': p_perm,
                'act_mean': act_d.mean(), 'sha_mean': sha_d.mean(),
                'act_dir': 'TOWARD' if act_d.mean() < 0 else 'AWAY',
                'sha_dir': 'TOWARD' if sha_d.mean() < 0 else 'AWAY',
            }

    # Enrichment
    enrichment = {}
    mean_roi_err = np.mean(np.array(mdd_r1_r), axis=0) if mdd_r1_r else None
    if mean_roi_err is not None:
        top_idx = np.argsort(mean_roi_err)[::-1]
        for k in [10, 15, 20, 30]:
            nc, ns, enr, ph = hypergeom_enrichment(top_idx[:k], circuit_mask, len(roi_meta))
            enrichment[f'top_{k}'] = {'k': nc, 'n': ns, 'enrichment': enr, 'p': ph}

    return {
        'd_sep': d_sep, 'p_sep': p_welch, 'p_overfit': p_overfit,
        'fp_rate': fp_rate, 'n_outliers': outlier_mask.sum(),
        'intervention': intervention, 'enrichment': enrichment,
        'train_time': train_time, 'mean_roi_err': mean_roi_err,
        'hc_te_z': hc_te_z, 'mdd_r1_z': mdd_r1_z, 'hc_ho_z': hc_ho_z,
        'gae': gae, 'history': history,
    }

# S9 — Main Experiment (v6 Coupled, Physics-Only Scoring)

In [10]:
# =============================================================================
# S9 --- MAIN EXPERIMENT
# =============================================================================

log.info('=' * 70)
log.info('MAIN EXPERIMENT: v6 coupled, physics-only scoring')
log.info('=' * 70)

main = run_experiment(
    encoder, hc_train_graphs, hc_test_graphs, hc_holdout_graphs,
    empirical_graphs, subjects_list, groups_map,
    use_coupled=True, scoring_fn=score_node_physics, verbose=True)

log.info('')
log.info('=== PRIMARY RESULTS ===')
log.info('  HC–MDD separation:     d=%+.3f, p=%.4e', main['d_sep'], main['p_sep'])
log.info('  Overfit check:         p=%.4f', main['p_overfit'])
log.info('  Holdout FP rate:       %.1f%%', 100 * main['fp_rate'])
log.info('  Outliers:              %d', main['n_outliers'])
for sn, iv in main['intervention'].items():
    log.info('  Intervention [%-13s]: d=%+.3f  perm_p=%.4f  Act:%s(%.4f) Sha:%s(%.4f)',
             sn, iv['d'], iv['perm_p'], iv['act_dir'], iv['act_mean'],
             iv['sha_dir'], iv['sha_mean'])
for k, v in main['enrichment'].items():
    log.info('  Enrichment [%s]: %d/%d (%.0f%%), %.2fx, p=%.4f',
             k, v['k'], v['n'], 100*v['k']/max(v['n'],1), v['enrichment'], v['p'])

[11:26:47] ======================================================================
[11:26:47] MAIN EXPERIMENT: v6 coupled, physics-only scoring
[11:26:47] ======================================================================
[11:26:47] GAE (coupled): trainable=586, d_z=6, σ=0.10, drop=0.3, λ_edge=0.50
[11:26:48]   Epoch   1 | Train 0.61486 | Test 0.52767 | Node 0.24559 | Edge 0.69472
[11:28:17]   Epoch  50 | Train 0.35307 | Test 0.35019 | Node 0.01118 | Edge 0.68367
[11:29:47]   Epoch 100 | Train 0.34345 | Test 0.34192 | Node 0.00934 | Edge 0.66806
[11:31:16]   Epoch 150 | Train 0.33759 | Test 0.33679 | Node 0.00849 | Edge 0.65797
[11:32:46]   Epoch 200 | Train 0.33723 | Test 0.33611 | Node 0.00839 | Edge 0.65743
[11:32:46] GAE training complete.
[11:32:50] 
[11:32:50] === PRIMARY RESULTS ===
[11:32:50]   HC–MDD separation:     d=+3.019, p=7.5176e-10
[11:32:50]   Overfit check:         p=0.8746
[11:32:50]   Holdout FP rate:       0.0%
[11:32:50]   Outliers:              1
[11:32:50]   

# S10 — Coupling Ablation: v5 (Decoupled) vs v6 (Coupled)

The central new analysis. Identical pipeline except `use_coupled=True` vs `False`.

In [11]:
# =============================================================================
# S10 --- COUPLING ABLATION
# =============================================================================

log.info('=' * 70)
log.info('COUPLING ABLATION: decoupled (v5) vs coupled (v6)')
log.info('=' * 70)

# ── v5 decoupled (edge decoders on frozen h) ──
log.info('\n--- v5 Decoupled ---')
m_decoupled = run_experiment(
    encoder, hc_train_graphs, hc_test_graphs, hc_holdout_graphs,
    empirical_graphs, subjects_list, groups_map,
    use_coupled=False, scoring_fn=score_node_physics, verbose=True)

# ── v6 coupled (already run as main) ──
m_coupled = main

log.info('\n=== COUPLING ABLATION RESULTS ===')
log.info('  %-12s  d_sep    WB_int_d  Sub_int_d  Enr_top10  Enr_top15', '')
log.info('  %-12s  %+.3f   %+.3f     %+.3f      %.2fx      %.2fx', 'Decoupled',
         m_decoupled['d_sep'],
         m_decoupled['intervention'].get('whole_brain', {}).get('d', 0),
         m_decoupled['intervention'].get('subcortical', {}).get('d', 0),
         m_decoupled['enrichment'].get('top_10', {}).get('enrichment', 0),
         m_decoupled['enrichment'].get('top_15', {}).get('enrichment', 0))
log.info('  %-12s  %+.3f   %+.3f     %+.3f      %.2fx      %.2fx', 'Coupled',
         m_coupled['d_sep'],
         m_coupled['intervention'].get('whole_brain', {}).get('d', 0),
         m_coupled['intervention'].get('subcortical', {}).get('d', 0),
         m_coupled['enrichment'].get('top_10', {}).get('enrichment', 0),
         m_coupled['enrichment'].get('top_15', {}).get('enrichment', 0))

# ROI ranking correlation between coupled and decoupled
if m_coupled['mean_roi_err'] is not None and m_decoupled['mean_roi_err'] is not None:
    from scipy.stats import spearmanr
    rho_cd, p_cd = spearmanr(m_coupled['mean_roi_err'], m_decoupled['mean_roi_err'])
    log.info('  ROI ranking correlation (coupled vs decoupled): rho=%.3f, p=%.4e', rho_cd, p_cd)

# Save
coupling_df = pd.DataFrame([
    {'architecture': 'decoupled_v5', 'd_sep': m_decoupled['d_sep'],
     'wb_d': m_decoupled['intervention'].get('whole_brain', {}).get('d', np.nan),
     'sub_d': m_decoupled['intervention'].get('subcortical', {}).get('d', np.nan),
     'enr_10': m_decoupled['enrichment'].get('top_10', {}).get('enrichment', np.nan)},
    {'architecture': 'coupled_v6', 'd_sep': m_coupled['d_sep'],
     'wb_d': m_coupled['intervention'].get('whole_brain', {}).get('d', np.nan),
     'sub_d': m_coupled['intervention'].get('subcortical', {}).get('d', np.nan),
     'enr_10': m_coupled['enrichment'].get('top_10', {}).get('enrichment', np.nan)},
])
coupling_df.to_csv(RESULTS_V6 / 'ablation_coupling.csv', index=False)

[11:32:50] ======================================================================
[11:32:50] COUPLING ABLATION: decoupled (v5) vs coupled (v6)
[11:32:50] ======================================================================
[11:32:50] 
--- v5 Decoupled ---
[11:32:50] GAE (decoupled): trainable=1882, d_z=6, σ=0.10, drop=0.3, λ_edge=0.50
[11:32:52]   Epoch   1 | Train 0.61251 | Test 0.51977 | Node 0.24711 | Edge 0.68701
[11:34:28]   Epoch  50 | Train 0.33441 | Test 0.33174 | Node 0.01090 | Edge 0.64690
[11:36:06]   Epoch 100 | Train 0.32995 | Test 0.32751 | Node 0.00849 | Edge 0.64270
[11:37:43]   Epoch 150 | Train 0.32847 | Test 0.32682 | Node 0.00808 | Edge 0.64057
[11:39:20]   Epoch 200 | Train 0.32871 | Test 0.32718 | Node 0.00803 | Edge 0.64114
[11:39:20] GAE training complete.
[11:39:25] 
=== COUPLING ABLATION RESULTS ===
[11:39:25]                 d_sep    WB_int_d  Sub_int_d  Enr_top10  Enr_top15
[11:39:25]   Decoupled     +2.750   +1.099     +1.406      2.19x      1.88x
[11:39:

# S11 — Scoring Method Comparison (Within Coupled Architecture)

In [12]:
# =============================================================================
# S11 --- SCORING METHOD COMPARISON
# =============================================================================

log.info('=' * 70)
log.info('SCORING METHOD COMPARISON (coupled v6 architecture)')
log.info('=' * 70)

gae_main = main['gae']

# Physics-only (primary — already computed)
d_physics = main['d_sep']

# All-7-feature scoring
m_7feat = run_experiment(
    encoder, hc_train_graphs, hc_test_graphs, hc_holdout_graphs,
    empirical_graphs, subjects_list, groups_map,
    use_coupled=True, scoring_fn=score_node_full, verbose=False)
d_7feat = m_7feat['d_sep']

# Edge-only scoring
def score_edge_only(gae_model, graph):
    _, result = score_node_physics(gae_model, graph)
    return score_edge_recon(gae_model, graph, result), result

hc_edge_s = np.array([score_edge_only(gae_main, g)[0].mean() for g in hc_test_graphs])
mdd_r1_list = [empirical_graphs[(s, 'rest1')] for s in subjects_list
               if (s, 'rest1') in empirical_graphs]
mdd_edge_s = np.array([score_edge_only(gae_main, g)[0].mean() for g in mdd_r1_list])
d_edge = cohens_d(mdd_edge_s, hc_edge_s)

# Mahalanobis in latent space
def compute_latent_stats(gae_model, graphs):
    z_means = []
    for g in graphs:
        g_c = prepare_graph_for_batching(g) if hasattr(g, 'subject') else g
        gae_model.eval()
        with torch.no_grad():
            z = gae_model(g_c)['z'].numpy()
        z_means.append(z.mean(axis=0))
    z_means = np.array(z_means)
    mu = z_means.mean(axis=0)
    cov = np.cov(z_means, rowvar=False) + np.eye(z_means.shape[1]) * 1e-6
    return mu, np.linalg.inv(cov), z_means

hc_mu_z, hc_cov_inv, _ = compute_latent_stats(gae_main, hc_train_graphs + hc_test_graphs)
hc_mahal = np.array([mahalanobis(z.mean(axis=0), hc_mu_z, hc_cov_inv)
                      for z in [gae_main(prepare_graph_for_batching(g))['z'].detach().numpy()
                                for g in hc_test_graphs]])
mdd_mahal = np.array([mahalanobis(z.mean(axis=0), hc_mu_z, hc_cov_inv)
                       for z in [gae_main(prepare_graph_for_batching(g))['z'].detach().numpy()
                                 for g in mdd_r1_list]])
d_mahal = cohens_d(mdd_mahal, hc_mahal)

# Per-feature decomposition
feature_names = ['a', 'omega', 'chisq', 's_plv', 's_mvar_in', 's_mvar_out', 'plv_within']
feature_ds = {}
for fi, fn in enumerate(feature_names):
    hc_f = np.array([((g.recon_target[:, fi].numpy() -
                       gae_main(prepare_graph_for_batching(g))['node_recon'][:, fi].detach().numpy()
                       ) ** 2).mean() for g in hc_test_graphs])
    mdd_f = np.array([((g.recon_target[:, fi].numpy() -
                        gae_main(prepare_graph_for_batching(g))['node_recon'][:, fi].detach().numpy()
                        ) ** 2).mean() for g in mdd_r1_list])
    feature_ds[fn] = cohens_d(mdd_f, hc_f)

scoring_summary = pd.DataFrame([
    {'Method': 'Physics-only (a,ω,χ²) [PRIMARY]', 'd_sep': d_physics},
    {'Method': 'All 7 features', 'd_sep': d_7feat},
    {'Method': 'Edge-only (z-based)', 'd_sep': d_edge},
    {'Method': 'Mahalanobis (latent)', 'd_sep': d_mahal},
])
scoring_summary.to_csv(RESULTS_V6 / 'scoring_comparison.csv', index=False)

log.info('\nScoring method comparison:')
for _, row in scoring_summary.iterrows():
    log.info('  %-35s d=%+.3f', row['Method'], row['d_sep'])

log.info('\nPer-feature d (reconstruction error):')
for fn, d in sorted(feature_ds.items(), key=lambda x: -abs(x[1])):
    log.info('  %-15s d=%+.3f', fn, d)

[11:39:25] ======================================================================
[11:39:25] SCORING METHOD COMPARISON (coupled v6 architecture)
[11:39:25] ======================================================================
[11:45:33] 
Scoring method comparison:
[11:45:33]   Physics-only (a,ω,χ²) [PRIMARY]     d=+3.019
[11:45:33]   All 7 features                      d=+3.640
[11:45:33]   Edge-only (z-based)                 d=-2.058
[11:45:33]   Mahalanobis (latent)                d=+2.998
[11:45:33] 
Per-feature d (reconstruction error):
[11:45:33]   chisq           d=+6.106
[11:45:33]   a               d=+5.911
[11:45:33]   s_mvar_in       d=+5.189
[11:45:33]   s_mvar_out      d=+4.969
[11:45:33]   omega           d=-1.760
[11:45:33]   plv_within      d=+1.435
[11:45:33]   s_plv           d=+1.287


# S12 — Permutation Null Distribution

In [13]:
# =============================================================================
# S12 --- PERMUTATION NULL
# =============================================================================

obs_d, perm_p, null_dist = permutation_test_d(
    main['mdd_r1_z'], main['hc_te_z'], n_perms=N_PERMUTATIONS)
log.info('Permutation test: observed d=%+.3f, p=%.6f (n=%d)', obs_d, perm_p, N_PERMUTATIONS)

[11:45:33] Permutation test: observed d=+3.019, p=0.000000 (n=10000)


# S13 — Seed Robustness

In [14]:
# =============================================================================
# S13 --- SEED ROBUSTNESS
# =============================================================================

log.info('=' * 70)
log.info('SEED ROBUSTNESS SWEEP (%d seeds)', N_SEED_RUNS)
log.info('=' * 70)

seed_results = []
for si in range(N_SEED_RUNS):
    s = SEED + si * 100
    log.info('  Seed %d/%d (seed=%d)', si + 1, N_SEED_RUNS, s)
    m = run_experiment(
        encoder, hc_train_graphs, hc_test_graphs, hc_holdout_graphs,
        empirical_graphs, subjects_list, groups_map,
        use_coupled=True, scoring_fn=score_node_physics, seed=s, verbose=False)
    row = {'seed': s, 'd_sep': m['d_sep'], 'fp_rate': m['fp_rate'],
           'n_outliers': m['n_outliers']}
    for sn, iv in m['intervention'].items():
        row[f'{sn}_d'] = iv['d']; row[f'{sn}_pp'] = iv['perm_p']
    for k, v in m['enrichment'].items():
        row[f'enr_{k}'] = v['enrichment']; row[f'enr_{k}_p'] = v['p']
    seed_results.append(row)

seed_df = pd.DataFrame(seed_results)
seed_df.to_csv(RESULTS_V6 / 'seed_sweep.csv', index=False)

log.info('\nSeed sweep summary:')
for col in ['d_sep', 'whole_brain_d', 'subcortical_d', 'enr_top_10', 'fp_rate']:
    if col in seed_df.columns:
        log.info('  %-20s mean=%+.3f  sd=%.3f  [%.3f, %.3f]',
                 col, seed_df[col].mean(), seed_df[col].std(),
                 seed_df[col].min(), seed_df[col].max())

[11:45:33] ======================================================================
[11:45:33] SEED ROBUSTNESS SWEEP (10 seeds)
[11:45:33] ======================================================================
[11:45:33]   Seed 1/10 (seed=42)
[11:51:35]   Seed 2/10 (seed=142)
[11:57:35]   Seed 3/10 (seed=242)
[12:03:36]   Seed 4/10 (seed=342)
[12:09:36]   Seed 5/10 (seed=442)
[12:15:40]   Seed 6/10 (seed=542)
[12:21:43]   Seed 7/10 (seed=642)
[12:27:45]   Seed 8/10 (seed=742)
[12:33:47]   Seed 9/10 (seed=842)
[12:39:50]   Seed 10/10 (seed=942)
[12:45:52] 
Seed sweep summary:
[12:45:52]   d_sep                mean=+2.911  sd=0.115  [2.744, 3.043]
[12:45:52]   whole_brain_d        mean=+1.150  sd=0.061  [1.092, 1.300]
[12:45:52]   subcortical_d        mean=+1.446  sd=0.080  [1.307, 1.613]
[12:45:52]   enr_top_10           mean=+1.847  sd=0.311  [1.252, 2.191]
[12:45:52]   fp_rate              mean=+0.000  sd=0.000  [0.000, 0.000]


# S14 — Bottleneck Dimension Sweep

In [15]:
# =============================================================================
# S14 --- BOTTLENECK SWEEP (coupled architecture)
# =============================================================================

log.info('=' * 70)
log.info('BOTTLENECK SWEEP (coupled v6)')
log.info('=' * 70)

bn_results = []
for dz in [4, 6, 8]:
    log.info('  d_z = %d', dz)
    m = run_experiment(
        encoder, hc_train_graphs, hc_test_graphs, hc_holdout_graphs,
        empirical_graphs, subjects_list, groups_map,
        latent_dim=dz, use_coupled=True, scoring_fn=score_node_physics, verbose=False)
    row = {'d_z': dz, 'd_sep': m['d_sep']}
    for sn, iv in m['intervention'].items():
        row[f'{sn}_d'] = iv['d']; row[f'{sn}_pp'] = iv['perm_p']
    for k, v in m['enrichment'].items():
        row[f'enr_{k}'] = v['enrichment']; row[f'enr_{k}_p'] = v['p']
    bn_results.append(row)
    log.info('    d_sep=%+.3f  wb_d=%+.3f  enr_top10=%.2fx',
             row['d_sep'], row.get('whole_brain_d', 0), row.get('enr_top_10', 0))

bn_df = pd.DataFrame(bn_results)
bn_df.to_csv(RESULTS_V6 / 'bottleneck_sweep.csv', index=False)
log.info('\n%s', bn_df.to_string(index=False))

[12:45:52] ======================================================================
[12:45:52] BOTTLENECK SWEEP (coupled v6)
[12:45:52] ======================================================================
[12:45:52]   d_z = 4
[12:51:46]     d_sep=+2.697  wb_d=+1.086  enr_top10=1.88x
[12:51:46]   d_z = 6
[12:57:47]     d_sep=+3.019  wb_d=+1.300  enr_top10=2.19x
[12:57:47]   d_z = 8
[13:03:52]     d_sep=+2.814  wb_d=+1.127  enr_top10=1.57x
[13:03:52] 
 d_z    d_sep  whole_brain_d  whole_brain_pp  circuit_d  circuit_pp  limbic_d  limbic_pp  subcortical_d  subcortical_pp  enr_top_10  enr_top_10_p  enr_top_15  enr_top_15_p  enr_top_20  enr_top_20_p  enr_top_30  enr_top_30_p
   4 2.697277       1.085595          0.0670   0.949957      0.0971  1.181280     0.0467       1.456742          0.0194    1.878261      0.058736    1.878261      0.019382    1.565217      0.061642    1.460870      0.051636
   6 3.019330       1.299552          0.0329   1.189064      0.0503  1.320565     0.0316       1.6

# S15 — Lambda Edge Sweep (Coupling Strength)

Unlike v5 where this had zero effect, in v6 lambda_edge controls
how strongly connectivity shapes z. This is now the critical hyperparameter.

In [16]:
# =============================================================================
# S15 --- LAMBDA EDGE SWEEP
# =============================================================================

log.info('=' * 70)
log.info('LAMBDA EDGE SWEEP (coupling strength)')
log.info('=' * 70)

le_results = []
for le in [0.0, 0.1, 0.25, 0.5, 1.0, 2.0]:
    log.info('  lambda_edge = %.2f', le)
    m = run_experiment(
        encoder, hc_train_graphs, hc_test_graphs, hc_holdout_graphs,
        empirical_graphs, subjects_list, groups_map,
        lambda_edge=le, use_coupled=True, scoring_fn=score_node_physics, verbose=False)
    row = {'lambda_edge': le, 'd_sep': m['d_sep']}
    for sn, iv in m['intervention'].items():
        row[f'{sn}_d'] = iv['d']
    for k, v in m['enrichment'].items():
        row[f'enr_{k}'] = v['enrichment']
    le_results.append(row)
    log.info('    d_sep=%+.3f  wb_d=%+.3f  enr_top10=%.2fx',
             row['d_sep'], row.get('whole_brain_d', 0), row.get('enr_top_10', 0))

le_df = pd.DataFrame(le_results)
le_df.to_csv(RESULTS_V6 / 'sweep_lambda_edge.csv', index=False)
log.info('\n%s', le_df.to_string(index=False))

[13:03:52] ======================================================================
[13:03:52] LAMBDA EDGE SWEEP (coupling strength)
[13:03:52] ======================================================================
[13:03:52]   lambda_edge = 0.00
[13:09:50]     d_sep=+2.750  wb_d=+1.100  enr_top10=2.19x
[13:09:50]   lambda_edge = 0.10
[13:15:51]     d_sep=+2.743  wb_d=+1.077  enr_top10=1.88x
[13:15:51]   lambda_edge = 0.25
[13:21:51]     d_sep=+2.806  wb_d=+1.124  enr_top10=1.88x
[13:21:51]   lambda_edge = 0.50
[13:27:50]     d_sep=+3.019  wb_d=+1.300  enr_top10=2.19x
[13:27:50]   lambda_edge = 1.00
[13:33:51]     d_sep=+2.962  wb_d=+1.160  enr_top10=2.19x
[13:33:51]   lambda_edge = 2.00
[13:39:49]     d_sep=+2.852  wb_d=+1.131  enr_top10=2.19x
[13:39:49] 
 lambda_edge    d_sep  whole_brain_d  circuit_d  limbic_d  subcortical_d  enr_top_10  enr_top_15  enr_top_20  enr_top_30
        0.00 2.750067       1.099573   1.012937  1.240872       1.407923    2.191304    1.878261    1.721739    1.

# S16 — Latent Space Analysis

In [17]:
# =============================================================================
# S16 --- LATENT SPACE ANALYSIS
# =============================================================================

log.info('=' * 70)
log.info('LATENT SPACE ANALYSIS')
log.info('=' * 70)

gae_main = main['gae']

def extract_latent(gae_model, graphs):
    z_means, z_nodes = [], []
    for g in graphs:
        g_c = prepare_graph_for_batching(g) if hasattr(g, 'subject') else g
        gae_model.eval()
        with torch.no_grad():
            z = gae_model(g_c)['z'].numpy()
        z_means.append(z.mean(axis=0)); z_nodes.append(z)
    return np.array(z_means), z_nodes

hc_z_means, hc_z_nodes = extract_latent(gae_main, hc_test_graphs)
mdd_z_means, mdd_z_nodes = extract_latent(gae_main, mdd_r1_list)

# UMAP
try:
    from umap import UMAP
    all_z = np.vstack([hc_z_means, mdd_z_means])
    labels = np.array(['HC'] * len(hc_z_means) + ['MDD'] * len(mdd_z_means))
    embedding = UMAP(n_components=2, n_neighbors=15, min_dist=0.1, random_state=SEED).fit_transform(all_z)
    umap_df = pd.DataFrame({'UMAP1': embedding[:, 0], 'UMAP2': embedding[:, 1], 'group': labels})
    umap_df.to_csv(RESULTS_V6 / 'umap_embeddings.csv', index=False)
    log.info('UMAP saved.')
except ImportError:
    umap_df = None; log.warning('UMAP not available.')

# Per-dimension separation
log.info('\nPer-latent-dimension d:')
for d in range(hc_z_means.shape[1]):
    log.info('  z_%d: d=%+.3f', d, cohens_d(mdd_z_means[:, d], hc_z_means[:, d]))

# Disentanglement
corr = np.corrcoef(np.vstack([hc_z_means, mdd_z_means]), rowvar=False)
off = corr[np.triu_indices_from(corr, k=1)]
log.info('\nDisentanglement: mean|r|=%.3f, max|r|=%.3f', np.abs(off).mean(), np.abs(off).max())

# Latent shift vs recon error
if main['mean_roi_err'] is not None:
    hc_roi_z = np.mean(hc_z_nodes, axis=0)
    mdd_roi_z = np.mean(mdd_z_nodes, axis=0)
    shift_mag = np.linalg.norm(mdd_roi_z - hc_roi_z, axis=1)
    from scipy.stats import spearmanr
    rho, p = spearmanr(shift_mag, main['mean_roi_err'])
    log.info('\nLatent shift vs recon error: rho=%.3f, p=%.4e', rho, p)

[13:39:49] ======================================================================
[13:39:49] LATENT SPACE ANALYSIS
[13:39:49] ======================================================================
[13:39:53] UMAP saved.
[13:39:53] 
Per-latent-dimension d:
[13:39:53]   z_0: d=+1.815
[13:39:53]   z_1: d=-1.928
[13:39:53]   z_2: d=+0.944
[13:39:53]   z_3: d=+1.264
[13:39:53]   z_4: d=-0.273
[13:39:53]   z_5: d=-1.264
[13:39:53] 
Disentanglement: mean|r|=0.663, max|r|=0.962
[13:39:53] 
Latent shift vs recon error: rho=-0.190, p=5.1237e-03


# S17 — Split-Half HC Validation

In [18]:
# =============================================================================
# S17 --- SPLIT-HALF VALIDATION
# =============================================================================

log.info('=' * 70)
log.info('SPLIT-HALF HC VALIDATION')
log.info('=' * 70)

n_half = len(hc_train_graphs) // 2
m_a = run_experiment(encoder, hc_train_graphs[:n_half], hc_test_graphs, hc_holdout_graphs,
                     empirical_graphs, subjects_list, groups_map,
                     use_coupled=True, scoring_fn=score_node_physics, verbose=False)
m_b = run_experiment(encoder, hc_train_graphs[n_half:], hc_test_graphs, hc_holdout_graphs,
                     empirical_graphs, subjects_list, groups_map,
                     use_coupled=True, scoring_fn=score_node_physics, verbose=False)

log.info('  Half A (n=%d): d_sep=%+.3f', n_half, m_a['d_sep'])
log.info('  Half B (n=%d): d_sep=%+.3f', len(hc_train_graphs) - n_half, m_b['d_sep'])
if m_a['mean_roi_err'] is not None and m_b['mean_roi_err'] is not None:
    from scipy.stats import spearmanr
    rho, p = spearmanr(m_a['mean_roi_err'], m_b['mean_roi_err'])
    log.info('  ROI ranking correlation: rho=%.3f, p=%.4e', rho, p)

[13:39:53] ======================================================================
[13:39:53] SPLIT-HALF HC VALIDATION
[13:39:53] ======================================================================
[13:47:17]   Half A (n=99): d_sep=+3.151
[13:47:17]   Half B (n=100): d_sep=+3.109
[13:47:17]   ROI ranking correlation: rho=0.998, p=2.0468e-269


# S18 — HC Test-Retest ICC

In [19]:
# =============================================================================
# S18 --- HC TEST-RETEST ICC
# =============================================================================

log.info('=' * 70)
log.info('HC TEST-RETEST ICC')
log.info('=' * 70)

hc_subj_map = defaultdict(list)
for info, g in zip(hc_file_info, hc_graphs):
    hc_subj_map[info['subject']].append((info['session'], g))

multi = {s: gs for s, gs in hc_subj_map.items() if len(gs) >= 2}
log.info('HC subjects with ≥2 sessions: %d', len(multi))

if len(multi) >= 5:
    subj_scores = {}
    for subj, sessions in multi.items():
        scores = []
        for sid, g in sorted(sessions, key=lambda x: x[0])[:2]:
            augment_graph(g, network_assignment)
            err, _ = score_node_physics(gae_main, g)
            scores.append(err.mean())
        subj_scores[subj] = scores

    all_sc = [{'subject': s, 'session': i, 'score': sc}
              for s, scs in subj_scores.items() for i, sc in enumerate(scs)]
    icc_df = pd.DataFrame(all_sc)
    sm = icc_df.groupby('subject')['score'].mean()
    ms_b = 2 * np.var(sm, ddof=1) if len(sm) > 1 else 0
    ms_w = max(icc_df.groupby('subject')['score'].var().mean(), 1e-12)
    icc = np.clip((ms_b - ms_w) / (ms_b + ms_w), -1, 1)
    log.info('  ICC(3,1) = %.3f (n=%d subjects)', icc, len(sm))

[13:47:17] ======================================================================
[13:47:17] HC TEST-RETEST ICC
[13:47:17] ======================================================================
[13:47:17] HC subjects with ≥2 sessions: 30
[13:47:17]   ICC(3,1) = 0.626 (n=30 subjects)


# S19 — Biological Interpretability

In [20]:
# =============================================================================
# S19 --- BIOLOGICAL INTERPRETABILITY
# =============================================================================

log.info('=' * 70)
log.info('BIOLOGICAL INTERPRETABILITY')
log.info('=' * 70)

# Enrichment curve
if main['mean_roi_err'] is not None:
    top_idx = np.argsort(main['mean_roi_err'])[::-1]
    enr_curve = []
    for k in range(5, 101, 5):
        nc, ns, enr, ph = hypergeom_enrichment(top_idx[:k], circuit_mask, len(roi_meta))
        enr_curve.append({'k': k, 'n_circuit': nc, 'enrichment': enr, 'p': ph})
    enr_curve_df = pd.DataFrame(enr_curve)
    enr_curve_df.to_csv(RESULTS_V6 / 'enrichment_curve.csv', index=False)
    log.info('Enrichment curve: peak %.2fx at top-%d',
             enr_curve_df['enrichment'].max(),
             enr_curve_df.loc[enr_curve_df['enrichment'].idxmax(), 'k'])

# Top ROIs
log.info('\nTop 15 anomalous ROIs:')
if main['mean_roi_err'] is not None:
    for rank, idx in enumerate(np.argsort(main['mean_roi_err'])[::-1][:15]):
        log.info('  %2d. %-45s %.6f  %s  %s', rank + 1,
                 roi_meta.iloc[idx]['roi_name'], main['mean_roi_err'][idx],
                 roi_meta.iloc[idx]['network'],
                 '[CIRCUIT]' if circuit_mask[idx] else '')

# Network-level
log.info('\nNetwork-level anomaly:')
for net in sorted(roi_meta['network'].unique()):
    mask = (roi_meta['network'] == net).values
    log.info('  %-25s %.6f (n=%d)', net, main['mean_roi_err'][mask].mean(), mask.sum())

# ROI intervention map
roi_int_act, roi_int_sha = [], []
for subj in subjects_list:
    r1k, r2k = (subj, 'rest1'), (subj, 'rest2')
    if r1k not in empirical_graphs or r2k not in empirical_graphs: continue
    e1, _ = score_node_physics(gae_main, empirical_graphs[r1k])
    e2, _ = score_node_physics(gae_main, empirical_graphs[r2k])
    delta = e2 - e1
    grp = groups_map.get(subj, '').lower()
    (roi_int_act if grp in ('active', 'experimental') else roi_int_sha).append(delta)

if len(roi_int_act) >= 2 and len(roi_int_sha) >= 2:
    roi_int_act = np.array(roi_int_act); roi_int_sha = np.array(roi_int_sha)
    roi_int_d = np.array([cohens_d(roi_int_act[:, j], roi_int_sha[:, j])
                           for j in range(len(roi_meta))])
    roi_int_df = pd.DataFrame({
        'roi_name': roi_meta['roi_name'], 'network': roi_meta['network'],
        'is_circuit': circuit_mask, 'intervention_d': roi_int_d,
        'act_mean_delta': roi_int_act.mean(axis=0),
        'sha_mean_delta': roi_int_sha.mean(axis=0),
    })
    roi_int_df.to_csv(RESULTS_V6 / 'roi_intervention_map.csv', index=False)

    log.info('\nTop 10 ROIs by |intervention d|:')
    for _, r in roi_int_df.reindex(
            roi_int_df['intervention_d'].abs().sort_values(ascending=False).index).head(10).iterrows():
        log.info('  %-45s d=%+.3f  %s  %s', r['roi_name'], r['intervention_d'],
                 r['network'], '[CIRCUIT]' if r['is_circuit'] else '')

# UKF correlation
mdd_anom, mdd_mean_a = [], []
for s in subjects_list:
    if (s, 'rest1') in empirical_graphs:
        g = empirical_graphs[(s, 'rest1')]
        err, _ = score_node_physics(gae_main, g)
        mdd_anom.append(err.mean()); mdd_mean_a.append(g.x[:, 0].numpy().mean())
if len(mdd_anom) >= 5:
    from scipy.stats import pearsonr, spearmanr
    rp, pp = pearsonr(mdd_anom, mdd_mean_a)
    rs, ps = spearmanr(mdd_anom, mdd_mean_a)
    log.info('\nUKF correlation: Pearson r=%.3f p=%.4f, Spearman r=%.3f p=%.4f', rp, pp, rs, ps)

# Amygdala
if main['mean_roi_err'] is not None and amyg_mask.any():
    d_amyg = cohens_d(main['mean_roi_err'][amyg_mask], main['mean_roi_err'][~amyg_mask])
    log.info('\nAmygdala: anomaly=%.6f (vs non=%.6f), d=%+.3f',
             main['mean_roi_err'][amyg_mask].mean(), main['mean_roi_err'][~amyg_mask].mean(), d_amyg)
    if len(roi_int_act) >= 2:
        d_amyg_int = cohens_d(roi_int_act[:, amyg_mask].mean(1), roi_int_sha[:, amyg_mask].mean(1))
        log.info('Amygdala intervention: d=%+.3f', d_amyg_int)

# Hemispheric
if main['mean_roi_err'] is not None:
    log.info('\nHemispheric: LH=%.6f, RH=%.6f, d=%+.3f',
             main['mean_roi_err'][lh_mask].mean(), main['mean_roi_err'][rh_mask].mean(),
             cohens_d(main['mean_roi_err'][lh_mask], main['mean_roi_err'][rh_mask]))
    if len(roi_int_act) >= 2:
        log.info('Intervention LH: d=%+.3f, RH: d=%+.3f',
                 cohens_d(roi_int_act[:, lh_mask].mean(1), roi_int_sha[:, lh_mask].mean(1)),
                 cohens_d(roi_int_act[:, rh_mask].mean(1), roi_int_sha[:, rh_mask].mean(1)))

# Heterogeneity
het_a, het_s, het_ac, het_sc = [], [], [], []
for subj in subjects_list:
    r1k, r2k = (subj, 'rest1'), (subj, 'rest2')
    if r1k not in empirical_graphs or r2k not in empirical_graphs: continue
    a1 = empirical_graphs[r1k].x[:, 0].numpy(); a2 = empirical_graphs[r2k].x[:, 0].numpy()
    ds = np.std(a2) - np.std(a1); dsc = np.std(a2[circuit_mask]) - np.std(a1[circuit_mask])
    grp = groups_map.get(subj, '').lower()
    if grp in ('active', 'experimental'):
        het_a.append(ds); het_ac.append(dsc)
    else:
        het_s.append(ds); het_sc.append(dsc)

if len(het_a) >= 2 and len(het_s) >= 2:
    het_a, het_s = np.array(het_a), np.array(het_s)
    d_het = cohens_d(het_a, het_s)
    t_het, p_het = sp_stats.ttest_ind(het_a, het_s, equal_var=False)
    log.info('\nHeterogeneity (raw a): WB d=%+.3f p=%.4f, Circuit d=%+.3f p=%.4f',
             d_het, p_het, cohens_d(np.array(het_ac), np.array(het_sc)),
             sp_stats.ttest_ind(het_ac, het_sc, equal_var=False)[1])

[13:47:17] ======================================================================
[13:47:17] BIOLOGICAL INTERPRETABILITY
[13:47:17] ======================================================================
[13:47:17] Enrichment curve: peak 2.50x at top-5
[13:47:17] 
Top 15 anomalous ROIs:
[13:47:17]    1. 7Networks_RH_Default_PFCdPFCm_4               0.062975  Default Mode  [CIRCUIT]
[13:47:17]    2. 7Networks_LH_Limbic_TempPole_1                0.045064  Limbic  [CIRCUIT]
[13:47:17]    3. 7Networks_LH_Limbic_TempPole_2                0.041219  Limbic  [CIRCUIT]
[13:47:17]    4. 7Networks_LH_Cont_Cing_2                      0.038579  Frontoparietal  
[13:47:17]    5. 7Networks_LH_Default_Temp_5                   0.038464  Default Mode  [CIRCUIT]
[13:47:17]    6. 7Networks_LH_Default_Par_1                    0.037587  Default Mode  
[13:47:17]    7. 7Networks_RH_SalVentAttn_FrOperIns_1          0.037126  Salience/VentAttn  
[13:47:17]    8. NAcc-rh                                       0.0

# S20 — Report Figures

In [21]:
# =============================================================================
# S20 --- REPORT FIGURES
# =============================================================================

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

IMG = RESULTS_V6 / 'img'
IMG.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({'figure.dpi': 150, 'savefig.dpi': 300, 'font.size': 10,
                     'figure.facecolor': 'white'})

# F1: HC vs MDD distributions
fig, ax = plt.subplots(figsize=(7, 4))
for data, pos, color, label in [(main['hc_te_z'], 0, '#4C72B0', 'HC'),
                                 (main['mdd_r1_z'], 1, '#DD8452', 'MDD')]:
    parts = ax.violinplot([data], positions=[pos], showmeans=True, showmedians=True)
    for pc in parts['bodies']:
        pc.set_facecolor(color); pc.set_alpha(0.7)
ax.set_xticks([0, 1])
ax.set_xticklabels([f'HC (n={len(main["hc_te_z"])})', f'MDD (n={len(main["mdd_r1_z"])})'])
ax.set_ylabel('Anomaly Score (z)'); ax.set_title(f'd={main["d_sep"]:+.3f}, p={main["p_sep"]:.2e}')
fig.tight_layout(); fig.savefig(IMG / 'fig_hc_mdd.png'); fig.savefig(IMG / 'fig_hc_mdd.pdf')
plt.close(fig); log.info('-> fig_hc_mdd')

# F2: Permutation null
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(null_dist, bins=80, density=True, color='#999', alpha=0.7)
ax.axvline(obs_d, color='#DD8452', lw=2, ls='--', label=f'd={obs_d:+.3f}')
ax.set_xlabel("Cohen's d"); ax.set_ylabel('Density')
ax.set_title(f'Permutation Null (p={perm_p:.6f})'); ax.legend()
fig.tight_layout(); fig.savefig(IMG / 'fig_perm_null.png'); fig.savefig(IMG / 'fig_perm_null.pdf')
plt.close(fig); log.info('-> fig_perm_null')

# F3: Intervention multiscale
if main['intervention']:
    scales = list(main['intervention'].keys())
    ds = [main['intervention'][s]['d'] for s in scales]
    ps = [main['intervention'][s]['perm_p'] for s in scales]
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(range(len(scales)), ds, color=['#4C72B0', '#55A868', '#C44E52', '#8172B2'])
    ax.set_xticks(range(len(scales))); ax.set_xticklabels(scales, rotation=15)
    ax.set_ylabel("Cohen's d"); ax.set_title('Multi-Scale Intervention Effect')
    for i, (d, p) in enumerate(zip(ds, ps)):
        ax.text(i, d + 0.05, f'd={d:+.2f}\np={p:.3f}{"*" if p < 0.05 else ""}',
                ha='center', va='bottom', fontsize=8)
    fig.tight_layout(); fig.savefig(IMG / 'fig_intervention.png'); fig.savefig(IMG / 'fig_intervention.pdf')
    plt.close(fig); log.info('-> fig_intervention')

# F4: Enrichment curve
if 'enr_curve_df' in dir():
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 4))
    a1.plot(enr_curve_df['k'], enr_curve_df['enrichment'], 'o-', color='#C44E52')
    a1.axhline(1, color='gray', ls='--', alpha=0.5); a1.set_xlabel('Top-k'); a1.set_ylabel('Enrichment')
    a2.semilogy(enr_curve_df['k'], enr_curve_df['p'], 'o-', color='#4C72B0')
    a2.axhline(0.05, color='gray', ls='--', alpha=0.5); a2.set_xlabel('Top-k'); a2.set_ylabel('p')
    fig.tight_layout(); fig.savefig(IMG / 'fig_enrichment.png'); fig.savefig(IMG / 'fig_enrichment.pdf')
    plt.close(fig); log.info('-> fig_enrichment')

# F5: Coupling ablation
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(['Decoupled (v5)', 'Coupled (v6)'],
       [m_decoupled['d_sep'], m_coupled['d_sep']],
       color=['#999', '#4C72B0'])
ax.set_ylabel("Cohen's d"); ax.set_title('Coupling Ablation: HC–MDD Separation')
for i, d in enumerate([m_decoupled['d_sep'], m_coupled['d_sep']]):
    ax.text(i, d + 0.05, f'{d:+.3f}', ha='center', fontsize=10)
fig.tight_layout(); fig.savefig(IMG / 'fig_coupling_ablation.png'); fig.savefig(IMG / 'fig_coupling_ablation.pdf')
plt.close(fig); log.info('-> fig_coupling_ablation')

# F6: Lambda edge sweep
if len(le_df) > 0:
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(le_df['lambda_edge'], le_df['d_sep'], 'o-', color='#4C72B0')
    ax.set_xlabel('λ_edge'); ax.set_ylabel("d_sep"); ax.set_title('Edge Coupling Strength')
    fig.tight_layout(); fig.savefig(IMG / 'fig_lambda_edge.png'); fig.savefig(IMG / 'fig_lambda_edge.pdf')
    plt.close(fig); log.info('-> fig_lambda_edge')

# F7: UMAP
if umap_df is not None:
    fig, ax = plt.subplots(figsize=(7, 5))
    for grp, c, m in [('HC', '#4C72B0', 'o'), ('MDD', '#DD8452', 's')]:
        mask = umap_df['group'] == grp
        ax.scatter(umap_df.loc[mask, 'UMAP1'], umap_df.loc[mask, 'UMAP2'],
                   c=c, marker=m, s=40, alpha=0.7, label=grp, edgecolors='white', lw=0.3)
    ax.set_xlabel('UMAP 1'); ax.set_ylabel('UMAP 2'); ax.legend()
    ax.set_title('Latent Space (v6 coupled)')
    fig.tight_layout(); fig.savefig(IMG / 'fig_umap.png'); fig.savefig(IMG / 'fig_umap.pdf')
    plt.close(fig); log.info('-> fig_umap')

# F8: Seed robustness
if len(seed_df) > 1:
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    for ax, col, title in zip(axes, ['d_sep', 'whole_brain_d', 'enr_top_10'],
                               ['HC–MDD d', 'Intervention d', 'Top-10 Enrichment']):
        if col in seed_df.columns:
            ax.boxplot(seed_df[col].dropna())
            ax.scatter(np.ones(len(seed_df)), seed_df[col], alpha=0.5, s=30, color='#4C72B0')
            ax.set_title(f'{title}\n(μ={seed_df[col].mean():.2f}±{seed_df[col].std():.2f})')
            ax.set_xticks([])
    fig.suptitle(f'Seed Robustness (n={N_SEED_RUNS})')
    fig.tight_layout(); fig.savefig(IMG / 'fig_seeds.png'); fig.savefig(IMG / 'fig_seeds.pdf')
    plt.close(fig); log.info('-> fig_seeds')

# F9: Training curves
h = main['history']
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 4))
a1.plot(h['epoch'], h['train_loss'], label='Train'); a1.plot(h['epoch'], h['test_loss'], label='Test')
a1.set_xlabel('Epoch'); a1.set_ylabel('Loss'); a1.set_title('GAE Loss'); a1.legend()
a2.plot(h['epoch'], h['node_recon'], label='Node', color='#4C72B0')
a2.plot(h['epoch'], h['edge_recon'], label='Edge', color='#DD8452')
a2.set_xlabel('Epoch'); a2.set_ylabel('Loss'); a2.set_title('Node vs Edge'); a2.legend()
fig.tight_layout(); fig.savefig(IMG / 'fig_training.png'); fig.savefig(IMG / 'fig_training.pdf')
plt.close(fig); log.info('-> fig_training')

# F10: Circuit enrichment bar
if main['mean_roi_err'] is not None:
    ti = np.argsort(main['mean_roi_err'])[::-1][:30]
    fig, ax = plt.subplots(figsize=(10, 6))
    colors = ['#C44E52' if circuit_mask[i] else '#999' for i in ti]
    names = [roi_meta.iloc[i]['roi_name'][:35] for i in ti]
    ax.barh(range(len(ti)), [main['mean_roi_err'][i] for i in ti], color=colors)
    ax.set_yticks(range(len(ti))); ax.set_yticklabels(names, fontsize=7); ax.invert_yaxis()
    ax.set_xlabel('Mean Anomaly'); ax.set_title('Top 30 ROIs (red = circuit)')
    fig.tight_layout(); fig.savefig(IMG / 'fig_roi_ranking.png'); fig.savefig(IMG / 'fig_roi_ranking.pdf')
    plt.close(fig); log.info('-> fig_roi_ranking')

log.info('All figures saved to %s', IMG)

[13:47:18] -> fig_hc_mdd
[13:47:18] -> fig_perm_null
[13:47:18] -> fig_intervention
[13:47:19] -> fig_enrichment
[13:47:19] -> fig_coupling_ablation
[13:47:19] -> fig_lambda_edge
[13:47:19] -> fig_umap
[13:47:19] -> fig_seeds
[13:47:19] -> fig_training
[13:47:19] -> fig_roi_ranking
[13:47:19] All figures saved to results/hopf_stgnn/v6/img


# S21 — Summary Export

In [22]:
# =============================================================================
# S21 --- SUMMARY EXPORT
# =============================================================================

log.info('=' * 70)
log.info('SUMMARY EXPORT')
log.info('=' * 70)

rows = [{'test': 'HC vs MDD separation', 'scale': 'whole_brain',
         'd': main['d_sep'], 'p': main['p_sep'],
         'method': 'Welch t (physics-only, coupled v6)', 'perm_p': perm_p}]
for sn, iv in main['intervention'].items():
    rows.append({'test': f'Intervention ({sn})', 'scale': sn,
                 'd': iv['d'], 'p': iv['p'], 'method': 'Welch t + perm', 'perm_p': iv['perm_p']})
for k, v in main['enrichment'].items():
    rows.append({'test': f'Enrichment ({k})', 'scale': k,
                 'd': v['enrichment'], 'p': v['p'], 'method': 'Hypergeometric'})

summary_df = pd.DataFrame(rows)
summary_df.to_csv(RESULTS_V6 / 'summary_statistics_v6.csv', index=False)
log.info('\n%s', summary_df.to_string(index=False))

# Anomaly scores
export = []
for i, s in enumerate(main['hc_te_z']):
    export.append({'cohort': 'HC_test', 'group': 'HC', 'session': 'avg', 'score': float(s)})
for i, s in enumerate(subjects_list):
    if i < len(main['mdd_r1_z']):
        export.append({'cohort': 'MDD', 'group': groups_map.get(s, ''), 'session': 'rest1',
                       'subject': s, 'score': float(main['mdd_r1_z'][i])})
pd.DataFrame(export).to_csv(RESULTS_V6 / 'anomaly_scores_v6.csv', index=False)

# ROI map
if main['mean_roi_err'] is not None:
    rm = roi_meta[['roi_name', 'network', 'is_depression_circuit']].copy()
    rm['mean_anomaly'] = main['mean_roi_err']
    rm.sort_values('mean_anomaly', ascending=False).to_csv(RESULTS_V6 / 'roi_anomaly_map_v6.csv', index=False)

# Model
torch.save(main['gae'].state_dict(), RESULTS_V6 / 'hopf_gae_v6.pt')
pd.DataFrame(main['history']).to_csv(RESULTS_V6 / 'gae_train_history.csv', index=False)

log.info('')
log.info('=' * 70)
log.info('v6 EXPERIMENT COMPLETE')
log.info('=' * 70)
log.info('Artifacts: %s', RESULTS_V6)

[13:47:19] ======================================================================
[13:47:19] SUMMARY EXPORT
[13:47:19] ======================================================================
[13:47:19] 
                      test       scale        d            p                             method  perm_p
      HC vs MDD separation whole_brain 3.019330 7.517639e-10 Welch t (physics-only, coupled v6)  0.0000
Intervention (whole_brain) whole_brain 1.299552 1.705559e-02                     Welch t + perm  0.0329
    Intervention (circuit)     circuit 1.189064 2.423794e-02                     Welch t + perm  0.0503
     Intervention (limbic)      limbic 1.320565 1.972767e-02                     Welch t + perm  0.0316
Intervention (subcortical) subcortical 1.612840 6.541844e-03                     Welch t + perm  0.0103
       Enrichment (top_10)      top_10 2.191304 1.334366e-02                     Hypergeometric     NaN
       Enrichment (top_15)      top_15 2.295652 8.081218e-04          